# VOX-SYNAPSE — Stage 1 Qwen smoke test

Этот notebook клонирует репозиторий, запускает локальные тесты, затем **явно** загружает Qwen3-1.7B в Colab и сохраняет диагностический архив. Перед запуском выбери GPU runtime.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
TOOL_LATENCY_MS = 1500  # @param {type:"integer"}

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL перед запуском")

In [ ]:
import os, platform, subprocess, sys

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("GPU не доступен. Выбери Runtime → Change runtime type → GPU")
except ImportError:
    print("PyTorch будет установлен на следующем шаге")

In [ ]:
from pathlib import Path

repo_dir = Path("/content/vox")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)

os.chdir(repo_dir)
print("Repository:", repo_dir)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml]"], check=True)
print("Dependencies installed")

In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage1"
artifacts.mkdir(parents=True, exist_ok=True)
tests_log = artifacts / "tests.log"
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
tests_log.write_text(test_run.stdout, encoding="utf-8")
print(test_run.stdout)
if test_run.returncode != 0:
    raise RuntimeError(f"Unit tests failed; see {tests_log}")

## Запуск модели

Только следующая ячейка разрешает скачивание весов. Все предыдущие шаги работают без модели.

In [ ]:
model_log = artifacts / "model_run.log"
command = [
    sys.executable, "-m", "vox.experiments.colab_stage1",
    "--allow-download",
    "--model", MODEL_ID,
    "--tool-latency-ms", str(TOOL_LATENCY_MS),
    "--output-dir", str(artifacts),
]
model_run = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
model_log.write_text(model_run.stdout, encoding="utf-8")
print(model_run.stdout)
print("Model process exit code:", model_run.returncode)
print("Даже при ошибке следующая ячейка упакует диагностические файлы.")

In [ ]:
import json

report_path = artifacts / "report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Status:", report.get("status"))
    print("Error:", report.get("error", ""))
    print("Result:", report.get("result_text", ""))
    print("Model load ms:", report.get("model_load_ms"))
    print("Runtime ms:", report.get("runtime_ms"))
else:
    print("report.json отсутствует; model_run.log всё равно будет в архиве")

In [ ]:
import shutil
from google.colab import files

archive_base = Path("/content/vox-colab-stage1-logs")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=artifacts)
print("Created:", archive_path)
files.download(archive_path)